Load Gold Delta

In [0]:
from pyspark.sql.functions import *

gold_path = "/Volumes/smart_fraud_databricks/default/raw_data/gold"

gold_df = spark.read.format("delta").load(gold_path)

print("Gold records:", gold_df.count())

display(gold_df.limit(10))

Gold records: 20051


txn_id,account_id,txn_date,amount,merchant,customer_name,account_type,credit_limit,branch,fraud_type,flagged_date,is_watchlisted,amount_to_credit_ratio,is_high_value
TXN011985,ACC0410,2025-06-29,129367.97,ATM_Withdrawal,Customer_410,Savings,200000.0,Delhi,null,null,0,0.64683985,0
TXN014943,ACC0476,2025-03-16,26517.62,BigBasket,Customer_476,Savings,500000.0,Pune,null,null,0,0.05303524,0
TXN014800,ACC0159,2025-07-15,99817.38,BigBasket,Customer_159,Salary,200000.0,Chennai,null,null,0,0.4990869,0
TXN000878,ACC0466,2025-03-30,72273.67,Zomato,Customer_466,Current,100000.0,Pune,null,null,0,0.7227367,0
TXN018533,ACC0438,2025-04-01,26084.84,BigBasket,Customer_438,Savings,500000.0,Mumbai,null,null,0,0.05216968,0
TXN015224,ACC0233,2025-05-24,109450.52,Unknown_POS,Customer_233,Current,100000.0,Chennai,Account Takeover,2025-05-09,1,1.0945052,1
TXN014385,ACC0455,2025-07-05,49937.67,Flipkart,Customer_455,Salary,500000.0,Mumbai,null,null,0,0.09987533999999999,0
TXN007293,ACC0452,2025-07-12,100834.15,Flipkart,Customer_452,Savings,200000.0,Bangalore,null,null,0,0.50417075,0
TXN013059,ACC0349,2025-05-27,56120.37,Unknown_POS,Customer_349,Salary,500000.0,Pune,Card Skimming,2025-05-25,1,0.11224074,0
TXN004388,ACC0187,2025-07-15,387.61,Swiggy,Customer_187,Savings,100000.0,Chennai,null,null,0,0.0038761,0


Check the fraud labels

In [0]:
display(
    gold_df.groupBy("is_watchlisted")
           .count()
           .orderBy("is_watchlisted")
)

is_watchlisted,count
0,18366
1,1685


Select ML features

In [0]:
ml_df = gold_df.select(
    "amount",
    "credit_limit",
    "amount_to_credit_ratio",
    "is_high_value",
    "is_watchlisted"
)

In [0]:
ml_df = ml_df.dropna(
    subset=[
        "amount",
        "credit_limit",
        "amount_to_credit_ratio",
        "is_watchlisted"
    ]
)

print("ML records:", ml_df.count())

ML records: 20046


Convert to Pandas

In [0]:
print("Training dataset size:", ml_df.count())

Training dataset size: 20046


Prepare Spark ML features

In [0]:
from pyspark.ml.feature import VectorAssembler

feature_columns = [
    "amount",
    "credit_limit",
    "amount_to_credit_ratio",
    "is_high_value"
]

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

model_df = assembler.transform(ml_df).select(
    "features",
    col("is_watchlisted").alias("label")
)

display(model_df.limit(10))

features,label
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""129367.97"",""200000.0"",""0.64683985"",""0.0""]}",0
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""26517.62"",""500000.0"",""0.05303524"",""0.0""]}",0
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""99817.38"",""200000.0"",""0.4990869"",""0.0""]}",0
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""72273.67"",""100000.0"",""0.7227367"",""0.0""]}",0
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""26084.84"",""500000.0"",""0.05216968"",""0.0""]}",0
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""109450.52"",""100000.0"",""1.0945052"",""1.0""]}",1
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""49937.67"",""500000.0"",""0.09987533999999999"",""0.0""]}",0
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""100834.15"",""200000.0"",""0.50417075"",""0.0""]}",0
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""56120.37"",""500000.0"",""0.11224074"",""0.0""]}",1
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""387.61"",""100000.0"",""0.0038761"",""0.0""]}",0


Split training and testing data

In [0]:
train_df, test_df = model_df.randomSplit(
    [0.8, 0.2],
    seed=42
)

print("Training records:", train_df.count())
print("Testing records:", test_df.count())

Training records: 16094
Testing records: 3952


Train a Random Forest model

In [0]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=100,
    seed=42
)

rf_model = rf.fit(train_df)

print("Random Forest model trained successfully!")

Random Forest model trained successfully!


Generate predictions

In [0]:
predictions = rf_model.transform(test_df)

display(
    predictions.select(
        "label",
        "prediction",
        "probability"
    ).limit(20)
)

label,prediction,probability
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9187601728357315"",""0.08123982716426853""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9187601728357315"",""0.08123982716426853""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9187601728357315"",""0.08123982716426853""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9187601728357315"",""0.08123982716426853""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9187601728357315"",""0.08123982716426853""]}"
1,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9187601728357315"",""0.08123982716426853""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9187601728357315"",""0.08123982716426853""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9187601728357315"",""0.08123982716426853""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9187601728357315"",""0.08123982716426853""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9187601728357315"",""0.08123982716426853""]}"


Evaluate the model

In [0]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

accuracy = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
).evaluate(predictions)

f1 = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
).evaluate(predictions)

print("Accuracy:", accuracy)
print("F1 Score:", f1)

Accuracy: 0.9046052631578947
F1 Score: 0.8592968821016271


Calculate Precision and Recall

In [0]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

precision = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision"
).evaluate(predictions)

recall = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
).evaluate(predictions)

f1 = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
).evaluate(predictions)

accuracy = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
).evaluate(predictions)

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")

Accuracy  : 0.9046
Precision : 0.8183
Recall    : 0.9046
F1 Score  : 0.8593


Create a confusion matrix

In [0]:
confusion_matrix = predictions.groupBy(
    "label",
    "prediction"
).count().orderBy(
    "label",
    "prediction"
)

display(confusion_matrix)

label,prediction,count
0,0.0,3575
1,0.0,377


Save predictions as a Gold/Prediction Delta table

In [0]:
prediction_path = "/Volumes/smart_fraud_databricks/default/raw_data/gold_predictions_delta"

print(prediction_path)

/Volumes/smart_fraud_databricks/default/raw_data/gold_predictions_delta


In [0]:
predictions.select(
    "label",
    "prediction",
    "probability"
).write.format("delta").mode("overwrite").save(prediction_path)

print("SUCCESS: Fraud predictions saved as Delta")

SUCCESS: Fraud predictions saved as Delta


In [0]:
saved_predictions = spark.read.format("delta").load(prediction_path)

print("Prediction records:", saved_predictions.count())

display(saved_predictions.limit(20))

Prediction records: 3952


label,prediction,probability
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9187601728357315"",""0.08123982716426853""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9187601728357315"",""0.08123982716426853""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9187601728357315"",""0.08123982716426853""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9187601728357315"",""0.08123982716426853""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9187601728357315"",""0.08123982716426853""]}"
1,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9187601728357315"",""0.08123982716426853""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9187601728357315"",""0.08123982716426853""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9187601728357315"",""0.08123982716426853""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9187601728357315"",""0.08123982716426853""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9187601728357315"",""0.08123982716426853""]}"


Check fraud predictions

In [0]:
saved_predictions = spark.read.format("delta").load(
    prediction_path
)

display(saved_predictions.limit(20))

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8977817292198587>, line 5
      1 saved_predictions = spark.read.format("delta").load(
      2     prediction_path
      3 )
----> 5 display(saved_predictions.limit(20))

File /databricks/python_shell/lib/dbruntime/display.py:136, in Display.display(self, input, *args, **kwargs)
    134     pass
    135 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 136     self.display_connect_table(input, **kwargs)
    137 elif isinstance(input, ConnectDataFrame):
    138     if input.isStreaming:

File /databricks/python_shell/lib/dbruntime/display.py:96, in Display.display_connect_table(self, df, **kwargs)
     91 except Exception as e:
     92     raise type(
     93         e
     94     )("IPython shell encountered an error or was missing data, please restart the notebook or contact Databricks support"

Save the trained model

In [0]:
model_path = "/Volumes/smart_fraud_databricks/default/raw_data/gold/fraud_model"

rf_model.write().overwrite().save(model_path)

print("Random Forest model saved successfully!")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8977817292198589>, line 3
      1 model_path = "/Volumes/smart_fraud_databricks/default/raw_data/gold/fraud_model"
----> 3 rf_model.write().overwrite().save(model_path)
      5 print("Random Forest model saved successfully!")

File /databricks/python/lib/python3.12/site-packages/pyspark/ml/connect/readwrite.py:52, in RemoteMLWriter.save(self, path)
     49 session = SparkSession.getActiveSession()
     50 assert session is not None
---> 52 RemoteMLWriter.saveInstance(
     53     self._instance,
     54     path,
     55     session,
     56     self.shouldOverwrite,
     57     self.optionMap,
     58 )

File /databricks/python/lib/python3.12/site-packages/pyspark/ml/connect/readwrite.py:97, in RemoteMLWriter.saveInstance(instance, path, session, shouldOverwrite, optionMap)
     95     command = pb2.Command()
     96     